In [ ]:
# --- Kalshi NBA spreads backpull + EV(t) + win candles (single cell) ---

import math
import time
from dataclasses import dataclass
from datetime import datetime, timezone, timedelta

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle

KALSHI_HOST = "https://api.elections.kalshi.com/trade-api/v2"

# ----------------------------
# CONFIG YOU EDIT
# ----------------------------
# Provide match tokens (base) + the team code you are betting *for*.
# If you only have full team market tickers like "KXNBAGAME-...-UTA", you can paste them and set team=None.
MATCHES = [
    {"match_token": "KXNBAGAME-26JAN19DALNYK", "team": "DAL", "p_win": 0.55},
]

# Which spread legs to backtest EV for (Kalshi strike_type is typically "greater"/"less"/"between").
# Defaults match your earlier example: "wins by >3.5" and "wins by <22.5".
SPREAD_LEGS = [("greater", 3.5), ("less", 22.5)]

# Margin model: assume point differential M ~ Normal(mu, sigma^2), with mu chosen so that P(M>0)=p_win.
SIGMA_POINTS = 12.0

# Candlestick interval in minutes (Kalshi supports 1, 60, 1440)
PERIOD_INTERVAL_MIN = 1

# How much history to pull, relative to close_time (keeps requests small)
HOURS_BEFORE_CLOSE = 30
HOURS_AFTER_CLOSE = 2

# ----------------------------
# Helpers
# ----------------------------

def _clamp(x, lo=1e-6, hi=1-1e-6):
    return max(lo, min(hi, x))

def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

# Acklam's inverse normal CDF approximation (fast, no scipy)
def norm_ppf(p: float) -> float:
    p = _clamp(p)
    # Coefficients in rational approximations
    a = [-3.969683028665376e+01,  2.209460984245205e+02,
         -2.759285104469687e+02,  1.383577518672690e+02,
         -3.066479806614716e+01,  2.506628277459239e+00]
    b = [-5.447609879822406e+01,  1.615858368580409e+02,
         -1.556989798598866e+02,  6.680131188771972e+01,
         -1.328068155288572e+01]
    c = [-7.784894002430293e-03, -3.223964580411365e-01,
         -2.400758277161838e+00, -2.549732539343734e+00,
          4.374664141464968e+00,  2.938163982698783e+00]
    d = [ 7.784695709041462e-03,  3.224671290700398e-01,
          2.445134137142996e+00,  3.754408661907416e+00]

    plow = 0.02425
    phigh = 1 - plow

    if p < plow:
        q = math.sqrt(-2 * math.log(p))
        return (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
               ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    if phigh < p:
        q = math.sqrt(-2 * math.log(1 - p))
        return -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                 ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)

    q = p - 0.5
    r = q*q
    return (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5]) * q / \
           (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)

def iso_to_dt(s: str) -> datetime:
    # Kalshi timestamps are ISO strings; pandas handles most variants well.
    return pd.to_datetime(s, utc=True).to_pydatetime()

def dt_to_unix(dt: datetime) -> int:
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return int(dt.timestamp())

def req_json(path: str, params=None, timeout=15, retries=3):
    url = f"{KALSHI_HOST}{path}"
    last_err = None
    for _ in range(retries):
        try:
            r = requests.get(url, params=params or {}, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(0.4)
    raise RuntimeError(f"GET failed: {url} params={params} err={last_err}")

def parse_strike_points(mkt: dict) -> float | None:
    """
    Prefer functional_strike if it's numeric-ish. Otherwise use floor_strike/cap_strike.
    Heuristic for scaling: if floor/cap is an int >= 100, assume it's points*100.
    """
    fs = mkt.get("functional_strike")
    if fs is not None:
        try:
            return float(str(fs).strip())
        except Exception:
            pass

    # Fallbacks
    st = (mkt.get("strike_type") or "").lower()
    if st == "greater":
        v = mkt.get("floor_strike")
    elif st == "less":
        v = mkt.get("cap_strike")
    elif st == "between":
        # return mid; we handle between separately if present
        lo = mkt.get("floor_strike")
        hi = mkt.get("cap_strike")
        if lo is None or hi is None:
            return None
        v = 0.5 * (float(lo) + float(hi))
    else:
        v = mkt.get("floor_strike") or mkt.get("cap_strike")

    if v is None:
        return None

    v = float(v)
    if abs(v) >= 100 and abs(v) <= 20000 and float(int(v)) == v:
        # common encoding for half-points: 3.5 -> 350, 22.5 -> 2250
        return v / 100.0
    return v

def model_prob_from_pwin(strike_type: str, strike_pts: float, p_win: float, sigma: float) -> float:
    """
    Map p_win := P(M>0) to mu via mu = sigma * Phi^{-1}(p_win), then compute P(condition on M).
    """
    z = norm_ppf(_clamp(p_win))
    mu = sigma * z
    t = (strike_pts - mu) / sigma

    st = strike_type.lower()
    if st == "greater":
        return 1.0 - norm_cdf(t)
    if st == "less":
        return norm_cdf(t)
    return float("nan")

def to_market_mid_from_candle(c: dict) -> float | None:
    # Use price.close_dollars when available; otherwise try yes_bid/yes_ask close.
    for key in ("price", "yes_bid", "yes_ask"):
        obj = c.get(key) or {}
        v = obj.get("close_dollars")
        if v is not None:
            try:
                return float(v)
            except Exception:
                pass
    return None

def candles_to_df(candles: list[dict], label: str) -> pd.DataFrame:
    rows = []
    for c in candles:
        ts = c.get("end_period_ts")
        if ts is None:
            continue
        dt = datetime.fromtimestamp(int(ts), tz=timezone.utc)
        price = c.get("price") or {}
        rows.append({
            "t": dt,
            f"{label}_open": float(price.get("open_dollars")) if price.get("open_dollars") is not None else np.nan,
            f"{label}_high": float(price.get("high_dollars")) if price.get("high_dollars") is not None else np.nan,
            f"{label}_low":  float(price.get("low_dollars"))  if price.get("low_dollars")  is not None else np.nan,
            f"{label}_close":float(price.get("close_dollars"))if price.get("close_dollars") is not None else np.nan,
            f"{label}_mid":  to_market_mid_from_candle(c),
            f"{label}_vol":  float(c.get("volume_fp")) if c.get("volume_fp") is not None else np.nan,
        })
    df = pd.DataFrame(rows).sort_values("t").set_index("t")
    return df

def plot_candles(ax, df_ohlc: pd.DataFrame, prefix: str, title: str):
    o = df_ohlc[f"{prefix}_open"].to_numpy()
    h = df_ohlc[f"{prefix}_high"].to_numpy()
    l = df_ohlc[f"{prefix}_low"].to_numpy()
    c = df_ohlc[f"{prefix}_close"].to_numpy()
    xs = mdates.date2num(df_ohlc.index.to_pydatetime())

    if len(xs) >= 2:
        w = 0.8 * (xs[1] - xs[0])
    else:
        w = 0.8 / (24*60)  # ~0.8 minute

    # Draw wicks + bodies (no explicit colors; matplotlib defaults)
    ax.vlines(xs, l, h, linewidth=1)
    for x, oo, cc in zip(xs, o, c):
        if np.isnan(oo) or np.isnan(cc):
            continue
        y0 = min(oo, cc)
        height = abs(cc - oo)
        if height == 0:
            height = 1e-6
        ax.add_patch(Rectangle((x - w/2, y0), w, height, fill=False, linewidth=1))

    ax.set_title(title)
    ax.set_ylabel("Price ($)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M", tz=timezone.utc))
    ax.grid(True, linewidth=0.5)

# ----------------------------
# Main routine per match
# ----------------------------

def run_match(match_token: str, team: str | None, p_win_const: float | None):
    # 1) Resolve win market ticker
    if team is None and match_token.count("-") >= 2:
        # might already be a full market ticker with team suffix
        win_ticker = match_token
        # infer team as last segment
        team = match_token.split("-")[-1]
        base_token = "-".join(match_token.split("-")[:-1])
    else:
        base_token = match_token
        win_ticker = f"{match_token}-{team}"

    # 2) Spread event ticker heuristic: swap series prefix
    spread_event = base_token.replace("KXNBAGAME", "KXNBASPREAD")

    # 3) Pull metadata
    win_mkt = req_json(f"/markets/{win_ticker}").get("market", {})
    win_open = iso_to_dt(win_mkt["open_time"]) if win_mkt.get("open_time") else None
    win_close = iso_to_dt(win_mkt["close_time"]) if win_mkt.get("close_time") else None
    win_result = (win_mkt.get("result") or "").lower()

    spread_evt = req_json(f"/events/{spread_event}").get("event", {})
    spread_markets = spread_evt.get("markets", []) or []

    # 4) Find desired spread market tickers for this team + legs
    picked = {}
    for m in spread_markets:
        # team filter: prefer primary_participant_key; fallback to ticker suffix
        pk = (m.get("primary_participant_key") or "").upper()
        tkr = (m.get("ticker") or "")
        if team and (pk == team.upper() or tkr.endswith(f"-{team.upper()}")):
            st = (m.get("strike_type") or "").lower()
            k = parse_strike_points(m)
            if k is None:
                continue
            for leg_st, leg_k in SPREAD_LEGS:
                if st == leg_st.lower() and abs(k - float(leg_k)) <= 1e-3:
                    picked[(leg_st.lower(), float(leg_k))] = {
                        "ticker": tkr,
                        "strike_type": st,
                        "strike": float(leg_k),
                        "result": (m.get("result") or "").lower(),
                        "expiration_value": m.get("expiration_value"),
                        "settlement_ts": m.get("settlement_ts"),
                        "title": m.get("title") or tkr,
                    }

    if len(picked) == 0:
        raise RuntimeError(
            f"Could not find spread legs {SPREAD_LEGS} for team={team} in spread_event={spread_event}. "
            "Print spread_evt['markets'] titles/strikes to debug."
        )

    # 5) Decide candle window
    # Use win_close if present; otherwise fall back to event strike_date.
    anchor_close = win_close
    if anchor_close is None:
        sd = spread_evt.get("strike_date")
        anchor_close = iso_to_dt(sd) if sd else datetime.now(timezone.utc)

    end_dt = anchor_close + timedelta(hours=HOURS_AFTER_CLOSE)
    start_dt = anchor_close - timedelta(hours=HOURS_BEFORE_CLOSE)

    # 6) Pull candlesticks for win + picked spreads (batch endpoint)
    tickers = [win_ticker] + [info["ticker"] for info in picked.values()]
    params = {
        "market_tickers": ",".join(tickers),
        "start_ts": dt_to_unix(start_dt),
        "end_ts": dt_to_unix(end_dt),
        "period_interval": int(PERIOD_INTERVAL_MIN),
        "include_latest_before_start": "true",
    }
    candles_blob = req_json("/markets/candlesticks", params=params).get("markets", [])

    by_ticker = {x["market_ticker"]: x.get("candlesticks", []) for x in candles_blob}

    win_df = candles_to_df(by_ticker.get(win_ticker, []), "win")

    spread_dfs = {}
    for key, info in picked.items():
        df = candles_to_df(by_ticker.get(info["ticker"], []), f"spr_{key[0]}_{key[1]}")
        spread_dfs[key] = df

    # 7) Build aligned dataframe on union of timestamps
    dfs = [win_df] + list(spread_dfs.values())
    all_idx = dfs[0].index
    for d in dfs[1:]:
        all_idx = all_idx.union(d.index)

    panel = pd.DataFrame(index=all_idx).sort_index()
    panel = panel.join(win_df, how="left")

    for key, d in spread_dfs.items():
        panel = panel.join(d, how="left")

    # Forward-fill prices for continuity
    panel = panel.sort_index().ffill()

    # 8) Compute EV time series
    # p_win(t): either constant or win price close (interpreted as probability)
    if p_win_const is not None:
        p_win_series = pd.Series(p_win_const, index=panel.index)
    else:
        p_win_series = panel["win_mid"].astype(float)

    p_win_series = p_win_series.apply(lambda x: _clamp(float(x)) if pd.notna(x) else np.nan).ffill()

    ev_cols = []
    for (leg_st, leg_k), info in picked.items():
        pref = f"spr_{leg_st}_{leg_k}"
        price = panel[f"{pref}_mid"].astype(float)

        p_model = p_win_series.apply(lambda pw: model_prob_from_pwin(leg_st, leg_k, pw, SIGMA_POINTS))
        ev = p_model - price  # YES-buy EV per $1 notional, ignoring fees

        col = f"EV_{leg_st}_{leg_k}"
        panel[col] = ev
        ev_cols.append(col)

    panel["EV_sum"] = panel[ev_cols].sum(axis=1)

    # 9) Plot: EV (top) + win candles (bottom)
    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.3], hspace=0.25)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])

    # EV plot
    for col in ev_cols:
        ax1.plot(panel.index, panel[col], label=col)
    ax1.plot(panel.index, panel["EV_sum"], label="EV_sum", linewidth=2)
    ax1.axhline(0.0, linewidth=1)
    ax1.set_title(f"{base_token} | team={team} | EV over time (sigma={SIGMA_POINTS}, p_win={'const' if p_win_const is not None else 'win_price'})")
    ax1.set_ylabel("EV ($ per $1 notional)")
    ax1.grid(True, linewidth=0.5)
    ax1.legend(loc="best")

    # Win candles
    if len(win_df) > 0:
        plot_candles(ax2, win_df, "win", title=f"Win market candles: {win_ticker} (result={win_result or 'unknown'})")
        # Mark settlement if present
        st = win_mkt.get("settlement_ts")
        if st:
            sdt = iso_to_dt(st)
            ax2.axvline(sdt, linewidth=1)
        # Mark final outcome level (0 or 1) for quick visual sanity
        if win_result in ("yes", "no"):
            ax2.axhline(1.0 if win_result == "yes" else 0.0, linewidth=1)
    else:
        ax2.set_title(f"No candlesticks returned for win market: {win_ticker}")
        ax2.axis("off")

    plt.show()

    # 10) Print quick summary
    print(f"\n=== {base_token} team={team} ===")
    print(f"win_ticker: {win_ticker} | win_result: {win_result or 'unknown'}")
    print(f"spread_event: {spread_event}")
    for (leg_st, leg_k), info in picked.items():
        print(f"  leg {leg_st} {leg_k}: {info['ticker']} | result={info['result'] or 'unknown'} | expiration_value={info['expiration_value']}")

# ----------------------------
# Run all
# ----------------------------
if not MATCHES:
    print("Populate MATCHES first (see config block).")
else:
    for m in MATCHES:
        run_match(
            match_token=m["match_token"],
            team=m.get("team"),
            p_win_const=m.get("p_win"),
        )
